In [1]:
# Cài đặt thư viện

!pip install -q transformers==4.46.3 peft==0.10.0 accelerate datasets evaluate rouge_score sentencepiece

import torch
import os
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from google.colab import drive
import evaluate
from datasets import load_dataset
from tqdm.auto import tqdm

# Kết nối Drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.8 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
MODEL_NAME = "VietAI/vit5-large"
ADAPTER_PATH = "/content/drive/MyDrive/Final-Statistical-Learning/Summarization/vit5-summarization-adapter"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=False)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Đang nạp trọng số LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval() # Bật chế độ đánh giá (tắt dropout)
print("✅ Mô hình đã sẵn sàng để chấm điểm!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.17G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.17G [00:00<?, ?B/s]

Đang nạp trọng số LoRA...
✅ Mô hình đã sẵn sàng để chấm điểm!


In [3]:
NUM_SAMPLES = 10000

print(f"Đang tải tập Test của VietNews và chọn ngẫu nhiên {NUM_SAMPLES} mẫu...")
# Tải tập test
test_dataset = load_dataset("nam194/vietnews", split="test")

# Trộn ngẫu nhiên (seed cố định để báo cáo có thể tái lập được) và cắt lấy NUM_SAMPLES
test_subset = test_dataset.shuffle(seed=42).select(range(NUM_SAMPLES))

def clean_data(batch):
    # Chuyển đổi dữ liệu, xóa dấu '_'
    cleaned_articles = [doc.replace('_', ' ') for doc in batch["article"]]
    cleaned_abstracts = [abs.replace('_', ' ') for abs in batch["abstract"]]

    return {
        "article": cleaned_articles,
        "abstract": cleaned_abstracts
    }

print("Đang làm sạch dấu gạch dưới '_' ...")
test_subset = test_subset.map(clean_data, batched=True, batch_size=100)
print("✅ Đã chuẩn bị xong dữ liệu test!")

Đang tải tập Test của VietNews và chọn ngẫu nhiên 10000 mẫu...


README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00001-84acb79f6c6547(…):   0%|          | 0.00/170M [00:00<?, ?B/s]

data/validation-00000-of-00001-210cc51bf(…):   0%|          | 0.00/38.3M [00:00<?, ?B/s]

data/test-00000-of-00001-123f98d55067eb7(…):   0%|          | 0.00/38.8M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/22184 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22498 [00:00<?, ? examples/s]

Đang làm sạch dấu gạch dưới '_' ...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

✅ Đã chuẩn bị xong dữ liệu test!


In [4]:
# Tải bộ công cụ chấm điểm ROUGE
rouge_metric = evaluate.load("rouge")

# Đảm bảo cấu hình pad_token chính xác cho việc xử lý theo Batch
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

predictions = []
references = []

# Tách riêng danh sách để tối ưu hóa tốc độ nạp dữ liệu vào Batch
articles = [item["article"] for item in test_subset]
references = [item["abstract"] for item in test_subset]

# CẤU HÌNH BATCH SIZE: GPU T4 của Colab gánh tốt cấu hình này
# Nếu muốn an toàn tuyệt đối không lo tràn VRAM, bạn để là 8. Nếu muốn nhanh hơn có thể thử 16.
BATCH_SIZE = 8

print(f"🚀 Bắt đầu quá trình suy luận theo BATCH ({BATCH_SIZE} mẫu/lượt) trên {len(articles)} mẫu...")

# Chạy vòng lặp cắt lát theo từng batch
for i in tqdm(range(0, len(articles), BATCH_SIZE)):
    batch_articles = articles[i : i + BATCH_SIZE]
    input_texts = ["tóm tắt: " + art for art in batch_articles]

    # Tokenize cả nhóm cùng lúc (bắt buộc phải có padding=True)
    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        max_length=1024,
        padding=True,
        truncation=True
    ).to(model.device)

    # Sinh tóm tắt hàng loạt cho cả batch
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=200,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    # Giải mã nhanh kết quả của cả batch và lưu lại
    pred_summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(pred_summaries)

print("\n📊 Đang tính toán điểm ROUGE...")
# Tính điểm (chạy thư viện evaluate)
results = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=False)

print("="*50)
print("🏆 KẾT QUẢ ĐÁNH GIÁ (ROUGE SCORE)")
print("="*50)
print(f"📌 ROUGE-1 (Độ trùng lặp từng từ):     {results['rouge1'] * 100:.2f} %")
print(f"📌 ROUGE-2 (Độ trùng lặp cụm 2 từ):    {results['rouge2'] * 100:.2f} %")
print(f"📌 ROUGE-L (Độ trùng lặp chuỗi dài):   {results['rougeL'] * 100:.2f} %")
print(f"📌 ROUGE-Lsum (Điểm tổng hợp):         {results['rougeLsum'] * 100:.2f} %")
print("="*50)

🚀 Bắt đầu quá trình suy luận theo BATCH (8 mẫu/lượt) trên 10000 mẫu...


  0%|          | 0/1250 [00:00<?, ?it/s]


📊 Đang tính toán điểm ROUGE...
🏆 KẾT QUẢ ĐÁNH GIÁ (ROUGE SCORE)
📌 ROUGE-1 (Độ trùng lặp từng từ):     57.71 %
📌 ROUGE-2 (Độ trùng lặp cụm 2 từ):    26.87 %
📌 ROUGE-L (Độ trùng lặp chuỗi dài):   36.82 %
📌 ROUGE-Lsum (Điểm tổng hợp):         36.82 %
